In [1]:
# Grand Canonical Monte Carlo (Ag nanoparticle)
#
# This notebook is split into short cells so each simulation step
# is easier to follow: imports, geometry, MC region, moves,
# thermodynamic setup, ensemble construction, and run.

In [2]:
from ase.cluster import Octahedron
from mace.calculators import mace_mp

from mcpy.moves import DeletionMove, InsertionMove
from mcpy.moves.move_selector import MoveSelector
from mcpy.ensembles.grand_canonical_ensemble import GrandCanonicalEnsemble
from mcpy.cell import SphericalCell

/home/energystorage/miniconda3/envs/mcpy/lib/python3.14/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


## 1) Build the starting nanoparticle

`Octahedron('Ag', 6, 1)` creates an Ag cluster that will be the initial configuration for the simulation.

In [3]:
atoms = Octahedron('Ag', 6, 1)
atoms

Cluster(symbols='Ag140', pbc=False)

## 2) Define the simulation cell for insertion/deletion

`SphericalCell` wraps the particle in a spherical region (with vacuum padding) and controls where Monte Carlo insertion/deletion attempts are sampled.

In [4]:
scell = SphericalCell(
    atoms,
    vacuum=3,
    species_radii={'Ag': 2.947, 'O': 0},
    mc_sample_points=100_000,
)
scell

## 3) Choose the energy calculator

The Monte Carlo acceptance test needs energies. Here we use `mace_mp` (a pretrained MACE model) on GPU (`device='cuda'`).

In [5]:
from mcpy.calculators import BaseCalculator
calculator = mace_mp(device='cuda')
calculator = BaseCalculator(calculator, 50, 0.2)

Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /home/energystorage/.cache/mace/macempa0mediummodel
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


/home/energystorage/miniconda3/envs/mcpy/lib/python3.14/site-packages/mace/calculators/mace.py:199: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
INFO:root:CUDA version: 12.8, CUDA device: 0
INFO:root:Using head default out of  ['default']


## 4) Configure Monte Carlo moves

- We only sample oxygen (`species = ['O']`).
- `DeletionMove` tries removing O atoms.
- `InsertionMove` tries adding O atoms.
- `MoveSelector` picks between them using the provided weights.

In [6]:
species = ['O']

move_list = [
    [50, 50],
    [
        DeletionMove(scell, species=['O'], seed=43215423143),
        InsertionMove(scell, species=['O'], min_insert=0.5, seed=3675437856),
    ],
]

move_selector = MoveSelector(*move_list)
move_selector

## 5) Set thermodynamic conditions

`mu` values are chemical potentials (in metal units here), and `T` is temperature in Kelvin.
The `delta_mu_O` offset is used to shift oxygen potential for this run.

In [7]:
mus = {'Ag': -2.99, 'O': -4.91}
delta_mu_O = -0.4
mus['O'] += delta_mu_O
T = 500

mus, T

({'Ag': -2.99, 'O': -5.3100000000000005}, 500)

## 6) Build the GCMC ensemble object

This assembles geometry, allowed cell region, calculator, species list, move selector, and thermodynamic state into one simulation object.

In [8]:
gcmc = GrandCanonicalEnsemble(
    atoms=atoms,
    cells=[scell],
    calculator=calculator,
    mu=mus,
    units_type='metal',
    species=species,
    temperature=T,
    move_selector=move_selector,
)

gcmc

## 7) Run the simulation

This is the original long production run (`1_000_000` MC steps). For quick testing, temporarily reduce `n_steps` to something like `1_000` or `10_000`.

In [ ]:
n_steps = 1_000_00
gcmc.run(n_steps)

INFO:GrandCanonicalEnsemble:+-------------------------------------------------+
INFO:GrandCanonicalEnsemble:| Grand Canonical Ensemble Monte Carlo Simulation |
INFO:GrandCanonicalEnsemble:+-------------------------------------------------+
INFO:GrandCanonicalEnsemble:Simulation Parameters:
INFO:GrandCanonicalEnsemble:Temperature (K): 500
INFO:GrandCanonicalEnsemble:Chemical potentials: {'Ag': -2.99, 'O': -5.3100000000000005}
INFO:GrandCanonicalEnsemble:Starting simulation...

INFO:GrandCanonicalEnsemble:Step       N_atoms    Energy (eV)     Acceptance Ratios (Del, Ins)
INFO:GrandCanonicalEnsemble:------------------------------------------------------------
INFO:GrandCanonicalEnsemble:0          154        -422.749451     7.8%, 36.7%         
INFO:GrandCanonicalEnsemble:1          160        -455.313812     11.1%, 20.0%        
